# 01 · Baseline: persistence and new establishment births

The main result. Regress log new establishment births (2022) on the church
persistence index, conditioning on historical income, education, population, and
state fixed effects. **Manuscript: Table 2.**

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../data/cleaned_data_v3.csv")

# formula-safe names
df = df.rename(columns={
    "pct_highschool_or_more (1990)": "pct_hs_1990",
    "Pop 2010": "pop_2010",
    "Established firms 2022": "firms_2022",
    "Established firms 1989": "firms_1989",
})

cols = ["Index_v2", "income_1989_real_2023", "pct_hs_1990",
        "pop_2010", "firms_2022", "firms_1989"]
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")
df = df.dropna(subset=cols)
df = df[(df["firms_2022"] > 0) & (df["pop_2010"] > 0) & (df["firms_1989"] > 0)]

# log(1+x) keeps zero-birth counties; plain log for the population control
df["log_firms_2022"] = np.log1p(df["firms_2022"])
df["log_pop_2010"] = np.log(df["pop_2010"])

# z-score the historical controls
df[["income_1989_scaled", "pct_hs_1990_scaled"]] = StandardScaler().fit_transform(
    df[["income_1989_real_2023", "pct_hs_1990"]])

# persistence + historical controls + state fixed effects, HC3 robust errors
model = smf.ols(
    "log_firms_2022 ~ Index_v2 + income_1989_scaled + pct_hs_1990_scaled "
    "+ log_pop_2010 + C(State)",
    data=df).fit(cov_type="HC3")
print(model.summary())

### Regression diagnostics

Residual behaviour, normality, heteroskedasticity, and influential observations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from statsmodels.stats.diagnostic import het_breuschpagan

df["residuals"] = model.resid
df["fitted_values"] = model.fittedvalues

plt.figure(figsize=(8, 5))
plt.scatter(df["fitted_values"], df["residuals"], alpha=0.5)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Fitted values"); plt.ylabel("Residuals")
plt.title("Residuals vs fitted"); plt.show()

sns.histplot(df["residuals"], kde=True)
plt.title("Residual distribution"); plt.xlabel("Residual"); plt.show()

stats.probplot(df["residuals"], dist="norm", plot=plt)
plt.title("QQ plot of residuals"); plt.show()

# Breusch-Pagan test for heteroskedasticity
bp = het_breuschpagan(model.resid, model.model.exog)
print(dict(zip(["LM", "LM p", "F", "F p"], bp)))

# Cook's distance for influential counties
cooks = model.get_influence().cooks_distance[0]
plt.figure(figsize=(10, 5))
plt.stem(np.arange(len(cooks)), cooks, markerfmt=",")
plt.axhline(4 / len(df), color="red", linestyle="--", label="4/n threshold")
plt.xlabel("Observation"); plt.ylabel("Cook's distance"); plt.legend(); plt.show()